In [0]:
%run ../config/config

In [0]:
from datetime import datetime, timezone
from pyspark.sql import functions as F
from databricks.sdk import WorkspaceClient
from databricks.labs.dqx.engine import DQEngine

from pathlib import Path
import sys

SRC_PATH = str(Path.cwd().parent / "src")
sys.path.insert(0, SRC_PATH)

from dq_tests import (
    silver_checks,
    gold_products_checks,
    gold_brands_checks,
    gold_fact_checks,
    write_quarantine,
    run_dqx
)

gate_failures = []

spark.conf.set("spark.databricks.execution.timeout", "18000")

SILVER_TABLE_PATH = f"{CATALOG}.{SILVER_SCHEMA}.{SILVER_TABLE}"
GOLD_FACT_PRODUCT_ENTRIES = f"{CATALOG}.{GOLD_SCHEMA}.{GOLD_FACT_PRODUCT_ENTRIES}"
GOLD_DIM_BRANDS = f"{CATALOG}.{GOLD_SCHEMA}.{GOLD_DIM_BRANDS}"
GOLD_DIM_PRODUCTS = f"{CATALOG}.{GOLD_SCHEMA}.{GOLD_DIM_PRODUCTS}"

RECONCILIATION_QUARANTINE = f"{CATALOG}.{GOLD_SCHEMA}.{RECONCILIATION_QUARANTINE}"
SILVER_QUARANTINE = f"{CATALOG}.{SILVER_SCHEMA}.{SILVER_TABLE_QUARANTINE}"
GOLD_DIM_BRANDS_QUARANTINE = f"{CATALOG}.{GOLD_SCHEMA}.{GOLD_DIM_BRANDS_QUARANTINE}"
GOLD_DIM_PRODUCTS_QUARANTINE = f"{CATALOG}.{GOLD_SCHEMA}.{GOLD_DIM_PRODUCTS_QUARANTINE}"
GOLD_FACT_PRODUCT_ENTRIES_QUARANTINE = f"{CATALOG}.{GOLD_SCHEMA}.{GOLD_FACT_PRODUCT_ENTRIES_QUARANTINE}"

dq_engine = DQEngine(WorkspaceClient())

run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")

print(f"Data quality run: {run_id}")

In [0]:
def gate_check(
    check_name: str,
    failure_count: int,
):
    if failure_count > 0:
        gate_failures.append(
            f"{check_name}: {failure_count} failure(s)"
        )
        print(
            f"FAILED: {check_name} "
            f"({failure_count} failure(s))"
        )
    else:
        print(f"PASSED: {check_name}")

In [0]:
silver_all = spark.table(SILVER_TABLE_PATH)

silver_current = silver_all.filter(
    F.col("is_current") == F.lit(True)
)

silver_status_checks = [
    check for check in silver_checks()
    if check.name == "is_current_not_null"
]

silver_status_valid, silver_status_invalid = run_dqx(
    engine=dq_engine,
    df=silver_all,
    checks=silver_status_checks,
    source_table=SILVER_TABLE_PATH,
    quarantine_table=SILVER_QUARANTINE,
    run_id=run_id,
)

silver_current_checks = [
    check for check in silver_checks()
    if check.name != "is_current_not_null"
]

silver_valid, silver_invalid = run_dqx(
    engine=dq_engine,
    df=silver_current,
    checks=silver_current_checks,
    source_table=SILVER_TABLE_PATH,
    quarantine_table=SILVER_QUARANTINE,
    run_id=run_id,
)

print("Silver current rows:", silver_current.count())
print("Silver invalid current rows:", silver_invalid.count())
print("Silver invalid status rows:", silver_status_invalid.count())

gate_check(
    "Silver invalid current rows",
    silver_invalid.count(),
)

gate_check(
    "Silver invalid is_current rows",
    silver_status_invalid.count(),
)

In [0]:
gold_products = spark.table(GOLD_DIM_PRODUCTS)
gold_brands = spark.table(GOLD_DIM_BRANDS)
gold_fact = spark.table(GOLD_FACT_PRODUCT_ENTRIES)

gold_products_valid, gold_products_invalid = run_dqx(
    engine=dq_engine,
    df=gold_products,
    checks=gold_products_checks(),
    source_table=GOLD_DIM_PRODUCTS,
    quarantine_table=GOLD_DIM_PRODUCTS_QUARANTINE,
    run_id=run_id,
)

gold_brands_valid, gold_brands_invalid = run_dqx(
    engine=dq_engine,
    df=gold_brands,
    checks=gold_brands_checks(),
    source_table=GOLD_DIM_BRANDS,
    quarantine_table=GOLD_DIM_BRANDS_QUARANTINE,
    run_id=run_id,
)

gold_fact_valid, gold_fact_invalid = run_dqx(
    engine=dq_engine,
    df=gold_fact,
    checks=gold_fact_checks(),
    source_table=GOLD_FACT_PRODUCT_ENTRIES,
    quarantine_table=GOLD_FACT_PRODUCT_ENTRIES_QUARANTINE,
    run_id=run_id,
)

gate_check(
    "Gold dim_products DQ",
    gold_products_invalid.count()
)

gate_check(
    "Gold dim_brands DQ",
    gold_brands_invalid.count()
)

gate_check(
    "Gold fact_product_entries DQ",
    gold_fact_invalid.count()
)

In [0]:
orphan_products = (
    gold_fact
    .filter(F.col("product_id").isNotNull())
    .join(
        gold_products.select("product_id").distinct(),
        on="product_id",
        how="left_anti",
    )
)

orphan_brands = (
    gold_fact
    .filter(F.col("brand_id").isNotNull())
    .join(
        gold_brands.select("brand_id").distinct(),
        on="brand_id",
        how="left_anti",
    )
)

write_quarantine(
    invalid_df=orphan_products,
    source_table=GOLD_FACT_PRODUCT_ENTRIES,
    quarantine_table=GOLD_FACT_PRODUCT_ENTRIES_QUARANTINE,
    run_id=run_id,
)

write_quarantine(
    invalid_df=orphan_brands,
    source_table=GOLD_FACT_PRODUCT_ENTRIES,
    quarantine_table=GOLD_FACT_PRODUCT_ENTRIES_QUARANTINE,
    run_id=run_id,
)

print("Orphan product references:", orphan_products.count())
print("Orphan brand references:", orphan_brands.count())

gate_check(
    "Gold fact - dim_products orphan references",
    orphan_products.count()
)

gate_check(
    "Gold fact - dim_brands orphan references",
    orphan_brands.count()
)

In [0]:
silver_keys = (
    silver_current
    .filter(F.col("product_id").isNotNull())
    .select("product_id")
    .distinct()
)

gold_keys = (
    gold_products
    .filter(F.col("product_id").isNotNull())
    .select("product_id")
    .distinct()
)

missing_from_gold = silver_keys.join(
    gold_keys,
    on="product_id",
    how="left_anti",
)

unexpected_in_gold = gold_keys.join(
    silver_keys,
    on="product_id",
    how="left_anti",
)

print("Distinct current Silver products:", silver_keys.count())
print("Distinct Gold dimension products:", gold_keys.count())
print("Products missing from Gold:", missing_from_gold.count())
print("Unexpected products in Gold:", unexpected_in_gold.count())

def save_reconciliation_failures(
    df,
    check_name: str,
    run_id: str,
):
    if df.limit(1).count() == 0:
        print(f"{check_name}: passed")
        return

    failures = (
        df
        .withColumn("_dq_check", F.lit(check_name))
        .withColumn("_dq_run_id", F.lit(run_id))
        .withColumn("_dq_checked_at", F.current_timestamp())
    )

    (
        failures.write
        .format("delta")
        .mode("append")
        .option("mergeSchema", "true")
        .saveAsTable(RECONCILIATION_QUARANTINE)
    )

save_reconciliation_failures(
    missing_from_gold,
    "silver_products_missing_from_gold",
    run_id,
)

save_reconciliation_failures(
    unexpected_in_gold,
    "gold_products_not_in_current_silver",
    run_id,
)


gate_check(
    "Silver products missing from Gold",
    missing_from_gold.count(),
)

gate_check(
    "Gold products not present in current Silver",
    unexpected_in_gold.count(),
)

In [0]:
if gate_failures:
    print("DATA QUALITY GATE FAILED")
    print("=" * 60)

    for failure in gate_failures:
        print(f"  - {failure}")

    print("=" * 60)

    raise RuntimeError(
        "Data quality gate failed:\n"
        + "\n".join(f"  - {failure}" for failure in gate_failures)
    )

else:
    print("DATA QUALITY GATE PASSED")
    print("=" * 60)

In [0]:
%skip
%sql
ALTER TABLE dbr_dev_ua5816bd.gulidov_artem_silver.products_silver
ALTER COLUMN product_id SET NOT NULL;

ALTER TABLE dbr_dev_ua5816bd.gulidov_artem_silver.products_silver
ADD CONSTRAINT silver_product_id_not_blank
CHECK (length(trim(product_id)) > 0);

ALTER TABLE dbr_dev_ua5816bd.gulidov_artem_silver.products_silver
ADD CONSTRAINT silver_nova_group_range
CHECK (nova_group IS NULL OR nova_group BETWEEN 0 AND 4);

ALTER TABLE dbr_dev_ua5816bd.gulidov_artem_gold.dim_products
ALTER COLUMN product_id SET NOT NULL;

ALTER TABLE dbr_dev_ua5816bd.gulidov_artem_gold.dim_brands
ALTER COLUMN brand_id SET NOT NULL;

ALTER TABLE dbr_dev_ua5816bd.gulidov_artem_gold.fact_product_entries
ALTER COLUMN entry_id SET NOT NULL;

ALTER TABLE dbr_dev_ua5816bd.gulidov_artem_gold.fact_product_entries
ALTER COLUMN product_id SET NOT NULL